## CutSky graph — simple square slice

- **300×300 Mpc/h** square in the chosen plot plane (default **x–z**, axes 0 and 2)
- **20 Mpc/h** thick slab along the perpendicular axis (default **y**, axis 1)
- **All edges** with **both endpoints** inside that box (no subsampling)


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.collections import LineCollection

META_PATH = Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_delaunay_split_hemis_20260402_metadata.json')
# META_PATH = Path("/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_mock_alpha_23032026_metadata.json")
BASE = META_PATH.parent

with open(META_PATH) as f:
    meta = json.load(f)

POINTS_XYZ = BASE / meta["files"]["points_xyz"]
EDGES = BASE / meta["files"]["edges"]

print("points_xyz:", POINTS_XYZ)
print("edges:", EDGES)
print("n_points (meta):", meta["n_points"], "  n_edges (meta):", meta["n_edges"])

In [ ]:
# --- Simple bounded slice: 300×300 Mpc/h square + 20 Mpc/h thick slab ---
# This cell keeps ONLY nodes inside the slab and ONLY edges with both endpoints inside.
# It also has guardrails to prevent OOM if the slab is accidentally too large.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

RNG_SEED = 0
rng = np.random.default_rng(RNG_SEED)

SIDE = 300.0          # Mpc/h square side in (AX1, AX2)
SLAB_THICK = 25.0     # Mpc/h thickness along PERP_AXIS
AX1, AX2 = 1, 0       # plot plane axes (here y vs x)
PERP_AXIS = 2         # slab axis (here z)

# Center of the square in the plot plane (Mpc/h)
P0_CENTER = 0.0       # center for AX1
P1_CENTER = 1000.0    # center for AX2

# Center of the slab (Mpc/h); set None to auto-median
PERP_CENTER = None

EDGE_CHUNK = 5_000_000

# Guardrails (tune)
MAX_NODES_IN_REGION = 300_000
MAX_EDGES_IN_REGION = 5_000_000

# --- Load memmaps ---
xyz = np.load(POINTS_XYZ, mmap_mode='r')
edges = np.load(EDGES, mmap_mode='r')
N = xyz.shape[0]
M = edges.shape[0]
print('xyz:', xyz.shape, 'edges:', M)

# Choose slab center cheaply if needed
if PERP_CENTER is None:
    sub_n = min(2_000_000, N)
    sub = rng.choice(N, size=sub_n, replace=False)
    PERP_CENTER = float(np.median(np.asarray(xyz[sub, PERP_AXIS], dtype=np.float64)))

half_side = 0.5 * SIDE
half_slab = 0.5 * SLAB_THICK

P0_LO, P0_HI = P0_CENTER - half_side, P0_CENTER + half_side
P1_LO, P1_HI = P1_CENTER - half_side, P1_CENTER + half_side
S0, S1 = PERP_CENTER - half_slab, PERP_CENTER + half_slab

p0 = xyz[:, AX1]
p1 = xyz[:, AX2]
sp = xyz[:, PERP_AXIS]

in_region = (
    (sp >= S0) & (sp < S1)
    & (p0 >= P0_LO) & (p0 < P0_HI)
    & (p1 >= P1_LO) & (p1 < P1_HI)
)

n_in = int(in_region.sum())
print('square:', f'axis{AX1}[{P0_LO:.1f},{P0_HI:.1f})', f'axis{AX2}[{P1_LO:.1f},{P1_HI:.1f})')
print('slab:', f'axis{PERP_AXIS}[{S0:.1f},{S1:.1f})  center={PERP_CENTER:.1f}')
print('nodes in region:', n_in)

if n_in == 0:
    raise RuntimeError('No nodes selected: move P0_CENTER/P1_CENTER/PERP_CENTER or widen SIDE/SLAB_THICK.')
if n_in > MAX_NODES_IN_REGION:
    raise RuntimeError(f'Too many nodes ({n_in:,}) for safe plotting. Tighten the window or reduce SIDE/SLAB_THICK.')

# Build a boolean mask for fast edge filtering
selected = np.zeros(N, dtype=np.bool_)
selected[in_region] = True

# --- Collect all edges with both endpoints in the region ---
kept_u = []
kept_v = []
kept = 0
for start in range(0, M, EDGE_CHUNK):
    stop = min(start + EDGE_CHUNK, M)
    e = edges[start:stop]
    u = e[:, 0]
    v = e[:, 1]
    m = selected[u] & selected[v]
    k = int(m.sum())
    if k:
        kept_u.append(np.asarray(u[m], dtype=np.int32))
        kept_v.append(np.asarray(v[m], dtype=np.int32))
        kept += k
    if start == 0 or (start // EDGE_CHUNK) % 5 == 0:
        print(f'edges scanned {stop:,}/{M:,}  kept {kept:,}')
    if kept > MAX_EDGES_IN_REGION:
        raise RuntimeError(f'Too many edges in region (>{MAX_EDGES_IN_REGION:,}). Tighten window before plotting.')

u_all = np.concatenate(kept_u) if kept_u else np.empty(0, dtype=np.int32)
v_all = np.concatenate(kept_v) if kept_v else np.empty(0, dtype=np.int32)
print('edges in region (complete):', u_all.size)

# --- Plot edges (2D projection) ---
xy = np.asarray(xyz[:, [AX1, AX2]], dtype=np.float32)
seg = np.empty((u_all.size, 2, 2), dtype=np.float32)
seg[:, 0, :] = xy[u_all]
seg[:, 1, :] = xy[v_all]

fig, ax = plt.subplots(figsize=(10, 10), dpi=220)
lc = LineCollection(seg, colors=(0.15, 0.5, 1.0, 0.14), linewidths=0.35)
lc.set_rasterized(True)
ax.add_collection(lc)
ax.set_xlim(P0_LO, P0_HI)
ax.set_ylim(P1_LO, P1_HI)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel(f'axis {AX1} [Mpc/h]')
ax.set_ylabel(f'axis {AX2} [Mpc/h]')
ax.set_title(
    f'All edges in {SIDE:g}×{SIDE:g}×{SLAB_THICK:g} Mpc/h subvolume (complete)\n'
    f'nodes={n_in:,} edges={u_all.size:,}'
)
plt.tight_layout()
plt.show()


In [ ]:
# --- Eigenvalues in the same slab (requires annotated mock FITS aligned to graph node order) ---
# Graph node i matches row i after the same Y1|Y5 + BOX_INDEX!=-1 filter as `build_abacus_graph.py`.
#
# MODIFIED (2026-04-25): plot galaxies in *cube* coordinates like cell 2.
# Instead of using `xyz` (graph plotting frame), we map each selected node to
# (FILE_NUM, HALO_INDEX) from the annotated FITS, load halo x_com from halo_info,
# then plot `pos_cube = mod(x_com, box)`.

import fitsio
from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog

# Annotated CutSky with LAMBDA1/2/3 (must match the mock used for this graph build).
ANNOTATED_FITS = Path(
    # "/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb.fits"
    # "/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits"
    "/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs_24042026_rsmooth_6/cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_rs6_ngrid2048_thr0p2.fits"
)

# Must match how the FITS was annotated (same as cell 2)
HALO_INFO_DIR = Path(
    "/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info"
)
HALO_POS_FIELD = "x_com"

# Optional: subsample scattered points for speed (edges plot uses all edges in slab already).
EIG_SCATTER_MAX = 80_000
RNG_EIG = np.random.default_rng(RNG_SEED + 1)

# Sanity: graph build catalog vs annotated labels (same row order after filter).
try:
    print("metadata source_path:", meta.get("source_path"))
except NameError:
    print("WARN: run the metadata cell first so `meta` exists.")
print("ANNOTATED_FITS:", ANNOTATED_FITS)


def _col(table, name):
    names = table.dtype.names
    m = {n.upper(): n for n in names}
    if name.upper() not in m:
        raise KeyError(f"Missing column {name!r}; have {names[:25]}...")
    return m[name.upper()]


def load_aligned_rows_and_eigs(fits_path: Path, *, expected_n: int):
    tab = fitsio.read(str(fits_path))
    in_y1 = _col(tab, "IN_Y1")
    in_y5 = _col(tab, "IN_Y5")
    bi = _col(tab, "BOX_INDEX")

    fn_col = _col(tab, "FILE_NUM")
    hi_col = _col(tab, "HALO_INDEX")

    l1n = _col(tab, "LAMBDA1")
    l2n = _col(tab, "LAMBDA2")
    l3n = _col(tab, "LAMBDA3")

    mask = ((tab[in_y1] == 1) | (tab[in_y5] == 1)) & (tab[bi] != -1)
    n = int(mask.sum())
    if n != expected_n:
        raise ValueError(
            f"Filtered FITS rows {n:,} != graph nodes N={expected_n:,}. "
            "Wrong annotated file or filters differ from graph build."
        )

    file_num = np.asarray(tab[fn_col][mask], dtype=np.int32)
    halo_index = np.asarray(tab[hi_col][mask], dtype=np.int64)

    l1 = np.asarray(tab[l1n][mask], dtype=np.float64)
    l2 = np.asarray(tab[l2n][mask], dtype=np.float64)
    l3 = np.asarray(tab[l3n][mask], dtype=np.float64)

    return file_num, halo_index, l1, l2, l3


file_num_all, halo_index_all, lam1, lam2, lam3 = load_aligned_rows_and_eigs(
    ANNOTATED_FITS, expected_n=N
)

# Nodes inside the slab selection from the previous cell (`selected`).
idx = np.flatnonzero(selected)
if idx.size == 0:
    raise RuntimeError("No nodes in slab; widen SIDE/SLAB_THICK or centers.")

# Eigenvalues for those nodes (aligned by node index)
l1s = lam1[idx]
l2s = lam2[idx]
l3s = lam3[idx]

# --- Cube-frame positions for those nodes via halo linkage ---
# Load halo x_com and wrap to [0,box) so it matches cell 2.
# Derive box from the T-Web outputs (TWEB_DIR must be set in the T-Web cells).
try:
    slab_files = sorted(TWEB_DIR.glob("abacus_cactus_tweb_rank*.npz"))
    with np.load(slab_files[0]) as d0:
        box = float(d0["boxsize"])
except Exception:
    # Fall back to 2000 if TWEB_DIR is not defined in this notebook state.
    box = 2000.0

fn = file_num_all[idx]
hi = halo_index_all[idx]

pos = np.full((idx.size, 3), np.nan, dtype=np.float64)
for f in np.unique(fn):
    m = fn == f
    hp = HALO_INFO_DIR / f"halo_info_{int(f):03d}.asdf"
    cat = CompaSOHaloCatalog(
        str(hp),
        fields=[HALO_POS_FIELD],
        subsamples=False,
        convert_units=True,
        verbose=False,
        cleaned=False,
    )
    arr = np.asarray(cat.halos[HALO_POS_FIELD], dtype=np.float64)
    nh = arr.shape[0]
    hidx = hi[m]
    ok = (hidx >= 0) & (hidx < nh)
    rows = np.where(m)[0]
    if np.any(ok):
        pos[rows[ok]] = arr[hidx[ok]]

ok = np.isfinite(pos).all(axis=1)
if not np.all(ok):
    print(f"WARN: {np.count_nonzero(~ok):,}/{ok.size:,} nodes have invalid halo linkage; dropping from scatter")

pos_cube = np.mod(pos[ok], box)

# Use cube x/y for plotting (match cell 2)
p0s = pos_cube[:, AX1]
p1s = pos_cube[:, AX2]

# Also filter eigenvalues to the same ok mask
l1s, l2s, l3s = l1s[ok], l2s[ok], l3s[ok]

# Subsample scatter points if needed
if pos_cube.shape[0] > EIG_SCATTER_MAX:
    sub = RNG_EIG.choice(pos_cube.shape[0], size=EIG_SCATTER_MAX, replace=False)
    p0p, p1p = p0s[sub], p1s[sub]
    g1, g2, g3 = l1s[sub], l2s[sub], l3s[sub]
else:
    p0p, p1p = p0s, p1s
    g1, g2, g3 = l1s, l2s, l3s

# Use the eigenvalues we already loaded as color arrays
(g1, g2, g3) = (l1s, l2s, l3s)

# Cube-frame axis limits so points are visible
PAD_PLOT = 2.0
xlo = max(0.0, float(np.min(p1s) - PAD_PLOT))
xhi = min(box, float(np.max(p1s) + PAD_PLOT))
ylo = max(0.0, float(np.min(p0s) - PAD_PLOT))
yhi = min(box, float(np.max(p0s) + PAD_PLOT))


# Rebuild edge segments in-plane (still uses `xyz` from the graph frame).
# NOTE: edges are plotted for structural reference; they will NOT be in cube coords unless
# you also remap all nodes to cube coords. The scatter points *are* now cube coords.
xy_sel = np.asarray(xyz[:, [AX1, AX2]], dtype=np.float32)
seg_ref = np.empty((u_all.size, 2, 2), dtype=np.float32)
seg_ref[:, 0, :] = xy_sel[u_all]
seg_ref[:, 1, :] = xy_sel[v_all]

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), dpi=200, constrained_layout=True)

lc0 = LineCollection(seg_ref, colors=(0.15, 0.5, 1.0, 0.14), linewidths=0.3)
lc0.set_rasterized(True)
axes[0].add_collection(lc0)
axes[0].set_xlim(P0_LO, P0_HI)
axes[0].set_ylim(P1_LO, P1_HI)
axes[0].set_aspect("equal")
axes[0].set_xlabel(f"axis {AX1} [Mpc/h]")
axes[0].set_ylabel(f"axis {AX2} [Mpc/h]")
axes[0].set_title("Slab: graph edges\n(graph frame; see note)")

for ax, g, title in zip(
    axes[1:],
    (g1, g2, g3),
    (r"$\lambda_1$", r"$\lambda_2$", r"$\lambda_3$"),
):
    lc = LineCollection(seg_ref, colors=(0.25, 0.25, 0.28, 0.06), linewidths=0.2)
    lc.set_rasterized(True)
    ax.add_collection(lc)
    sc = ax.scatter(
        p0p,
        p1p,
        c=g,
        s=4,
        cmap="magma",
        alpha=0.85,
        rasterized=True,
        vmin=np.nanpercentile(g, 2),
        vmax=np.nanpercentile(g, 98),
    )
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
    # ax.set_xlim(P0_LO, P0_HI)
    # ax.set_ylim(P1_LO, P1_HI)
    ax.set_xlim(ylo, yhi)  # axis AX1 in cube coords
    ax.set_ylim(xlo, xhi)  # axis AX2 in cube coords
    ax.set_aspect("equal")
    ax.set_xlabel(f"cube axis {AX1} [0,{box:g})")
    ax.set_ylabel(f"cube axis {AX2} [0,{box:g})")
    ax.set_title(f"T-Web {title} (halo-linked cube coords)\n n={pos_cube.shape[0]:,} scatter n={p0p.size:,}")

fig.suptitle(
    f"Eigenvalue maps (cube coords)  |  annotated: {ANNOTATED_FITS.name}",
    fontsize=11,
)
plt.show()


In [ ]:
# Full T-Web λ fields over the same subvolume (grid avg over slab thickness)
# Robust to mixed coordinate conventions per-axis:
# - if an axis bounds already lie in [0,box), use as-is
# - if an axis bounds look centered (e.g. negative), shift by +box/2

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

TWEB_DIR = Path(
    "/pscratch/sd/d/dkololgi/AbacusSummit_densities/tweb_rank_outputs_fullgrid_v3/"
    "dens_AbacusSummit_base_c000_ph000_z0.200_ngrid2048_box2000_thr0p2/"
    "backend_optimized_ngrid_2048_rsmooth_6"   # <-- change rsmooth_XX
)

if (AX1, AX2, PERP_AXIS) != (1, 0, 2):
    raise RuntimeError(f"Assumes AX1,AX2,PERP_AXIS=(1,0,2), got {(AX1,AX2,PERP_AXIS)}")

slabs = sorted(TWEB_DIR.glob("abacus_cactus_tweb_rank*.npz"))
if not slabs:
    raise RuntimeError(f"No slabs in {TWEB_DIR}")

with np.load(slabs[0]) as d0:
    ngrid = int(d0["ngrid"])
    box = float(d0["boxsize"])
cell = box / ngrid
shift = 0.5 * box  # 1000 for box=2000

# Original bounds from your slice cell (in your plotting convention)
b0 = [None, None, None]
b1 = [None, None, None]
b0[AX1], b1[AX1] = float(P0_LO), float(P0_HI)
b0[AX2], b1[AX2] = float(P1_LO), float(P1_HI)
b0[PERP_AXIS], b1[PERP_AXIS] = float(S0), float(S1)

# Convert to T-Web grid coords per axis
def to_grid_bounds(lo, hi):
    # If already inside [0,box), don't touch.
    if 0.0 <= lo < hi <= box:
        return lo, hi, 0.0
    # If centered-ish, shift.
    lo2, hi2 = lo + shift, hi + shift
    return lo2, hi2, shift

grid_lo = [0.0, 0.0, 0.0]
grid_hi = [0.0, 0.0, 0.0]
used_shift = [0.0, 0.0, 0.0]
for a in range(3):
    grid_lo[a], grid_hi[a], used_shift[a] = to_grid_bounds(b0[a], b1[a])

# Hard sanity: after possible shift, must be in [0,box)
for a, name in enumerate(["x","y","z"]):
    if not (0.0 <= grid_lo[a] < grid_hi[a] <= box):
        raise RuntimeError(
            f"Axis {a} ({name}) bounds still outside [0,box): "
            f"orig=[{b0[a]},{b1[a]}], grid=[{grid_lo[a]},{grid_hi[a]}], box={box}"
        )

# Index ranges (half-open)
x0, x1 = int(np.floor(grid_lo[0] / cell)), int(np.ceil(grid_hi[0] / cell))
y0, y1 = int(np.floor(grid_lo[1] / cell)), int(np.ceil(grid_hi[1] / cell))
z0, z1 = int(np.floor(grid_lo[2] / cell)), int(np.ceil(grid_hi[2] / cell))
x0, x1 = max(0, x0), min(ngrid, x1)
y0, y1 = max(0, y0), min(ngrid, y1)
z0, z1 = max(0, z0), min(ngrid, z1)

nx, ny, nz = (x1 - x0), (y1 - y0), (z1 - z0)
print("per-axis applied shift:", used_shift, "(=1000 means that axis was centered)")
print(f"grid ranges: x[{x0},{x1}) y[{y0},{y1}) z[{z0},{z1}) -> (nx,ny,nz)=({nx},{ny},{nz})")

# x->slab map
ix2sid = np.full(ngrid, -1, np.int32)
slab_meta = []
for sid, fp in enumerate(slabs):
    with np.load(fp) as d:
        xs, xe = int(d["x_start"]), int(d["x_end"])
    ix2sid[xs:xe] = sid
    slab_meta.append((fp, xs, xe))

need = np.unique(ix2sid[x0:x1])
if np.any(need < 0):
    raise RuntimeError("Some x indices are not covered by slab files.")

# Output images are (y, x) since we plot y vs x (AX1=1 vertical, AX2=0 horizontal)
out = [np.zeros((ny, nx), dtype=np.float64) for _ in range(3)]

for sid in need:
    fp, xs, xe = slab_meta[int(sid)]
    gx0, gx1 = max(x0, xs), min(x1, xe)
    lx0, lx1 = gx0 - xs, gx1 - xs
    ox0, ox1 = gx0 - x0, gx1 - x0

    with np.load(fp) as d:
        ev = d["eig_vals"][:, lx0:lx1, y0:y1, z0:z1].astype(np.float64, copy=False)  # (3,nx_seg,ny,nz)

    avg = ev.mean(axis=3)  # avg over z -> (3, nx_seg, ny)
    for k in range(3):
        out[k][:, ox0:ox1] = avg[k].T  # (ny, nx_seg)

# Plot in your original plotting convention (x uses P1 bounds, y uses P0 bounds)
extent = (float(P1_LO), float(P1_HI), float(P0_LO), float(P0_HI))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=200, constrained_layout=True)
titles = (r"$\langle\lambda_1\rangle$", r"$\langle\lambda_2\rangle$", r"$\langle\lambda_3\rangle$")
for ax, img, t in zip(axes, out, titles):
    im = ax.imshow(img, origin="lower", extent=extent, cmap="magma", aspect="equal")
    ax.set_title(t)
    ax.set_xlabel("x [Mpc/h] (as plotted)")
    ax.set_ylabel("y [Mpc/h] (as plotted)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

plt.show()

In [ ]:
# --- Diagnose FITS vs T-Web: sample grid at halo x_com (must match annotate_cutsky_with_tweb_eigs.py) ---
# This re-generates the check you ran before, but FIXES the critical detail:
# we apply np.mod(pos_cube, box) before computing voxel indices, exactly like
# annotate_cutsky_with_tweb_eigs.to_grid_indices() does.

import numpy as np
from pathlib import Path
import fitsio
from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog

# Must match the smoothing scale used to build ANNOTATED_FITS
TWEB_DIR = Path(
    "/pscratch/sd/d/dkololgi/AbacusSummit_densities/tweb_rank_outputs_fullgrid_v3/"
    "dens_AbacusSummit_base_c000_ph000_z0.200_ngrid2048_box2000_thr0p2/"
    "backend_optimized_ngrid_2048_rsmooth_16"
)
HALO_INFO_DIR = Path("/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info")
HALO_POS_FIELD = "x_com"  # must match the annotation run

# --- helpers (reuse your _col if it exists) ---
def _col_local(table, name):
    m = {n.upper(): n for n in table.dtype.names}
    if name.upper() not in m:
        raise KeyError(f"Missing column {name!r}; have {table.dtype.names[:25]}...")
    return m[name.upper()]

_col_use = _col if "_col" in globals() else _col_local

# --- load annotated FITS and reproduce the same filter used for graph build ---
tab = fitsio.read(str(ANNOTATED_FITS))
in_y1 = _col_use(tab, "IN_Y1")
in_y5 = _col_use(tab, "IN_Y5")
bi = _col_use(tab, "BOX_INDEX")
mask = ((tab[in_y1] == 1) | (tab[in_y5] == 1)) & (tab[bi] != -1)
tab_f = tab[mask]

# Sanity: alignment assumption for your notebook cells
if tab_f.shape[0] != N:
    raise RuntimeError(f"Filtered FITS rows {tab_f.shape[0]:,} != graph nodes N={N:,}")

# Choose some node indices to test (inside your current slab selection, if available)
rng = np.random.default_rng(0)
if "selected" in globals():
    idx_all = np.flatnonzero(selected)
else:
    idx_all = np.arange(N, dtype=np.int64)

if idx_all.size == 0:
    raise RuntimeError("No indices to test (selected is empty).")

idx = idx_all if idx_all.size <= 8000 else rng.choice(idx_all, size=8000, replace=False)

# FITS eigenvalues for these nodes
lam_fits = np.stack([lam1[idx], lam2[idx], lam3[idx]], axis=-1).astype(np.float64)

# FILE_NUM / HALO_INDEX linkage for these nodes
file_num = np.asarray(tab_f[_col_use(tab_f, "FILE_NUM")][idx], dtype=np.int32)
halo_index = np.asarray(tab_f[_col_use(tab_f, "HALO_INDEX")][idx], dtype=np.int64)

# --- load halo cube positions x_com for these nodes ---
pos_cube = np.full((idx.size, 3), np.nan, dtype=np.float64)

for fn in np.unique(file_num):
    sel = file_num == fn
    hp = HALO_INFO_DIR / f"halo_info_{int(fn):03d}.asdf"
    cat = CompaSOHaloCatalog(
        str(hp),
        fields=[HALO_POS_FIELD],
        subsamples=False,
        convert_units=True,
        verbose=False,
        cleaned=False,
    )
    arr = np.asarray(cat.halos[HALO_POS_FIELD], dtype=np.float64)  # could be centered or [0,box)
    nh = arr.shape[0]
    hidx = halo_index[sel]
    ok = (hidx >= 0) & (hidx < nh)
    rows = np.where(sel)[0]
    if np.any(ok):
        pos_cube[rows[ok]] = arr[hidx[ok]]

ok = np.isfinite(pos_cube).all(axis=1)
pos_cube = pos_cube[ok]
lam_fits = lam_fits[ok]

print(f"valid halo-linked rows: {ok.sum():,}/{idx.size:,}")

# --- sample the T-Web slabs at these cube positions ---
slab_files = sorted(TWEB_DIR.glob("abacus_cactus_tweb_rank*.npz"))
if not slab_files:
    raise RuntimeError(f"No slabs in {TWEB_DIR}")

with np.load(slab_files[0]) as d0:
    ngrid = int(d0["ngrid"])
    box = float(d0["boxsize"])
cell = box / ngrid

# IMPORTANT: match annotation code path (periodic wrap)
pos_mod = np.mod(pos_cube, box)

ix = np.clip(np.floor(pos_mod[:, 0] / cell).astype(np.int32), 0, ngrid - 1)
iy = np.clip(np.floor(pos_mod[:, 1] / cell).astype(np.int32), 0, ngrid - 1)
iz = np.clip(np.floor(pos_mod[:, 2] / cell).astype(np.int32), 0, ngrid - 1)

# x->slab maps
ix_to_slab = np.full(ngrid, -1, dtype=np.int32)
slabs = []
for sid, fp in enumerate(slab_files):
    with np.load(fp) as d:
        xs, xe = int(d["x_start"]), int(d["x_end"])
    ix_to_slab[xs:xe] = sid
    slabs.append((fp, xs))

sid = ix_to_slab[ix]
if np.any(sid < 0):
    raise RuntimeError("Some x indices are not covered by slabs (unexpected).")

lam_grid = np.empty_like(lam_fits)
for s in np.unique(sid):
    m = sid == s
    fp, xs = slabs[int(s)]
    li = (ix[m] - xs).astype(np.int64)
    yj = iy[m].astype(np.int64)
    zk = iz[m].astype(np.int64)
    with np.load(fp) as d:
        ev = d["eig_vals"]  # (3, nx_local, ngrid, ngrid)
        lam_grid[m, 0] = ev[0, li, yj, zk]
        lam_grid[m, 1] = ev[1, li, yj, zk]
        lam_grid[m, 2] = ev[2, li, yj, zk]

diff = lam_grid - lam_fits
print("grid@mod(x_com) vs FITS (mean abs diff):", np.mean(np.abs(diff), axis=0))
for k in range(3):
    r = np.corrcoef(lam_grid[:, k], lam_fits[:, k])[0, 1]
    print(f"corr(grid@mod(x_com), FITS) lambda{k+1}:", r)

In [ ]:
# --- Full T-Web field for *this slab's galaxies* (map slab nodes -> halo x_com -> cube window) ---
# This uses the CURRENT slab selection (`selected` from cell 2), then:
# - maps node indices -> (FILE_NUM, HALO_INDEX) from ANNOTATED_FITS (after same Y1|Y5 & BOX_INDEX!=-1 mask)
# - loads halo cube-frame positions x_com from halo_info_*.asdf
# - applies periodic wrap: pos_cube = mod(x_com, box)
# - defines the cube window as the tight bounding box of these galaxies (with small padding)
# - slices the T-Web eig_vals on that cube window and averages over z thickness

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import fitsio
from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog

# Must match the annotated FITS you are using
HALO_INFO_DIR = Path("/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info")
HALO_POS_FIELD = "x_com"  # must match how the FITS was annotated

# Must match rsmooth used in ANNOTATED_FITS
TWEB_DIR = Path(
    "/pscratch/sd/d/dkololgi/AbacusSummit_densities/tweb_rank_outputs_fullgrid_v3/"
    "dens_AbacusSummit_base_c000_ph000_z0.200_ngrid2048_box2000_thr0p2/"
    "backend_optimized_ngrid_2048_rsmooth_6"
)

# Helpers

def _col_local(table, name):
    m = {n.upper(): n for n in table.dtype.names}
    if name.upper() not in m:
        raise KeyError(f"Missing column {name!r}; have {table.dtype.names[:25]}...")
    return m[name.upper()]

_col_use = _col if "_col" in globals() else _col_local

# Slab nodes in the graph frame
idx = np.flatnonzero(selected)
if idx.size == 0:
    raise RuntimeError("selected is empty; run the slice cell first.")
print("slab nodes (graph):", idx.size)

# Load annotated FITS and reproduce graph-build row order
_tab = fitsio.read(str(ANNOTATED_FITS))
in_y1 = _col_use(_tab, "IN_Y1")
in_y5 = _col_use(_tab, "IN_Y5")
bi = _col_use(_tab, "BOX_INDEX")
mask = ((_tab[in_y1] == 1) | (_tab[in_y5] == 1)) & (_tab[bi] != -1)
_tab_f = _tab[mask]
if _tab_f.shape[0] != N:
    raise RuntimeError(f"Filtered FITS rows {_tab_f.shape[0]:,} != graph nodes N={N:,}")

fn_col = _col_use(_tab_f, "FILE_NUM")
hi_col = _col_use(_tab_f, "HALO_INDEX")
file_num = np.asarray(_tab_f[fn_col][idx], dtype=np.int32)
halo_index = np.asarray(_tab_f[hi_col][idx], dtype=np.int64)

# Load halo cube positions
pos = np.full((idx.size, 3), np.nan, dtype=np.float64)
for fn in np.unique(file_num):
    m = file_num == fn
    hp = HALO_INFO_DIR / f"halo_info_{int(fn):03d}.asdf"
    cat = CompaSOHaloCatalog(
        str(hp),
        fields=[HALO_POS_FIELD],
        subsamples=False,
        convert_units=True,
        verbose=False,
        cleaned=False,
    )
    arr = np.asarray(cat.halos[HALO_POS_FIELD], dtype=np.float64)
    nh = arr.shape[0]
    hidx = halo_index[m]
    ok = (hidx >= 0) & (hidx < nh)
    rows = np.where(m)[0]
    if np.any(ok):
        pos[rows[ok]] = arr[hidx[ok]]

ok = np.isfinite(pos).all(axis=1)
pos = pos[ok]
print("valid halo-linked slab galaxies:", pos.shape[0], "/", idx.size)
if pos.shape[0] == 0:
    raise RuntimeError("No valid halo positions loaded.")

# Discover T-Web slabs
slab_files = sorted(TWEB_DIR.glob("abacus_cactus_tweb_rank*.npz"))
if not slab_files:
    raise RuntimeError(f"No T-Web slabs in {TWEB_DIR}")

with np.load(slab_files[0]) as d0:
    ngrid = int(d0["ngrid"])
    box = float(d0["boxsize"])
cell = box / ngrid

# Apply periodic wrap exactly like annotate_cutsky_with_tweb_eigs.py
a = np.mod(pos, box)

# Define cube bounding box of these galaxies (pad slightly)
PAD = 2.0  # Mpc/h padding around the min/max
lo = np.maximum(0.0, np.min(a, axis=0) - PAD)
hi = np.minimum(box, np.max(a, axis=0) + PAD)

# Convert to grid index ranges
ix0, ix1 = int(np.floor(lo[0] / cell)), int(np.ceil(hi[0] / cell))
iy0, iy1 = int(np.floor(lo[1] / cell)), int(np.ceil(hi[1] / cell))
iz0, iz1 = int(np.floor(lo[2] / cell)), int(np.ceil(hi[2] / cell))
ix0, ix1 = max(0, ix0), min(ngrid, ix1)
iy0, iy1 = max(0, iy0), min(ngrid, iy1)
iz0, iz1 = max(0, iz0), min(ngrid, iz1)

nx, ny, nz = ix1 - ix0, iy1 - iy0, iz1 - iz0
print(f"Cube window (Mpc/h): x[{lo[0]:.2f},{hi[0]:.2f}] y[{lo[1]:.2f},{hi[1]:.2f}] z[{lo[2]:.2f},{hi[2]:.2f}]")
print(f"Grid window: x[{ix0},{ix1}) y[{iy0},{iy1}) z[{iz0},{iz1}) -> (nx,ny,nz)=({nx},{ny},{nz})")

# Build x->slab map
ix_to_slab = np.full(ngrid, -1, dtype=np.int32)
slab_meta = []
for sid, fp in enumerate(slab_files):
    with np.load(fp) as d:
        xs, xe = int(d["x_start"]), int(d["x_end"])
    ix_to_slab[xs:xe] = sid
    slab_meta.append((fp, xs, xe))

need = np.unique(ix_to_slab[ix0:ix1])
if np.any(need < 0):
    raise RuntimeError("Some x indices not covered by slabs.")

# Extract full field and average over z thickness
img = [np.zeros((ny, nx), dtype=np.float64) for _ in range(3)]

for sid in need:
    fp, xs, xe = slab_meta[int(sid)]
    gx0, gx1 = max(ix0, xs), min(ix1, xe)
    lx0, lx1 = gx0 - xs, gx1 - xs
    ox0, ox1 = gx0 - ix0, gx1 - ix0

    with np.load(fp) as d:
        ev = d["eig_vals"][:, lx0:lx1, iy0:iy1, iz0:iz1].astype(np.float64, copy=False)  # (3,nx_seg,ny,nz)

    avg = ev.mean(axis=3)  # mean over z -> (3,nx_seg,ny)
    for k in range(3):
        img[k][:, ox0:ox1] = avg[k].T  # (ny,nx_seg)

# Plot cube x-y plane
extent = (lo[0], hi[0], lo[1], hi[1])
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=200, constrained_layout=True)
for k, (ax, im, title) in enumerate(zip(axes, img, (r"$\langle\lambda_1\rangle$", r"$\langle\lambda_2\rangle$", r"$\langle\lambda_3\rangle$"))):
    h = ax.imshow(im, origin="lower", extent=extent, cmap="magma", aspect="equal")
    ax.scatter(a[:, 0], a[:, 1], c="white", s=1.0, alpha=0.90, rasterized=True)
    ax.set_title(title + " (grid avg over z)\npoints = slab galaxies in cube frame")
    ax.set_xlabel("cube x [0,2000)")
    ax.set_ylabel("cube y [0,2000)")
    plt.colorbar(h, ax=ax, fraction=0.046, pad=0.02)
plt.show()

In [ ]:
# --- Full density field over the same cube window (true density grid) ---
# Uses your 2048^3 density cube and slices the same (ix0:ix1, iy0:iy1, iz0:iz1)
# window computed in the previous cell. Then averages over z thickness and overplots slab galaxies.

import numpy as np
import matplotlib.pyplot as plt

DENSITY_NPY = "/pscratch/sd/d/dkololgi/AbacusSummit_densities/density_fields/AbacusSummit_base_c000_ph000_z0.200_ngrid_2048_10pc_density_field.npy"

# Reuse from previous cell: ix0/ix1/iy0/iy1/iz0/iz1, lo/hi, a (cube positions of slab galaxies)
for name in ("ix0", "ix1", "iy0", "iy1", "iz0", "iz1", "lo", "hi", "a"):
    if name not in globals():
        raise RuntimeError(f"Missing `{name}` from previous cell; run the slab->x_com->cube window cell first.")

rho = np.load(DENSITY_NPY, mmap_mode="r")  # (2048,2048,2048) float32

# Expect axis order (x,y,z)
blk = np.asarray(rho[ix0:ix1, iy0:iy1, iz0:iz1], dtype=np.float32)
if blk.size == 0:
    raise RuntimeError("Empty density slice; check ix/iy/iz bounds.")

# Average over z -> (nx, ny) then transpose to (ny, nx) for imshow
mean_xy = blk.mean(axis=2).T

extent = (lo[0], hi[0], lo[1], hi[1])

fig, ax = plt.subplots(figsize=(7.5, 7.5), dpi=220)
img = np.log10(np.maximum(mean_xy, 1e-6))
im = ax.imshow(img, origin="lower", extent=extent, cmap="viridis", aspect="equal")
ax.scatter(a[:, 0], a[:, 1], c="red", s=1.0, alpha=0.9, rasterized=True)
ax.set_title("Density field: log10(mean over z thickness)\npoints=slab galaxies (cube frame)")
ax.set_xlabel("cube x [0,2000)")
ax.set_ylabel("cube y [0,2000)")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label("log10(density)")

plt.tight_layout()
plt.show()


In [ ]:
# Optional: pan/zoom the same edge set in Plotly (requires plotly)
try:
    import plotly.graph_objects as go
    xy_i = np.asarray(xyz[:, [AX1, AX2]], dtype=np.float32)
    xs = np.empty(u_all.size * 3, dtype=np.float32)
    ys = np.empty(u_all.size * 3, dtype=np.float32)
    xs[0::3] = xy_i[u_all, 0]
    ys[0::3] = xy_i[u_all, 1]
    xs[1::3] = xy_i[v_all, 0]
    ys[1::3] = xy_i[v_all, 1]
    xs[2::3] = np.nan
    ys[2::3] = np.nan
    fig = go.Figure(
        data=[go.Scattergl(x=xs, y=ys, mode="lines", line=dict(color="rgba(40,140,255,0.35)", width=1), hoverinfo="skip")]
    )
    fig.update_layout(
        width=900,
        height=900,
        title=f"Same slice — interactive  Edges={u_all.size:,}",
        xaxis=dict(title=f"axis {AX1}", range=[P0_LO, P0_HI]),
        yaxis=dict(title=f"axis {AX2}", range=[P1_LO, P1_HI], scaleanchor="x", scaleratio=1),
        template="plotly_white",
        margin=dict(l=50, r=20, t=60, b=50),
    )
    fig.show()
except Exception as e:
    print("Plotly not available or failed:", repr(e))


In [ ]:
# --- Survey-space segment (RA/DEC/Z) covering the same region as the cube subvolume above ---
# Uses the current slab selection (`selected`) and the same FITS->graph row alignment (`_tab_f`).

import numpy as np
import matplotlib.pyplot as plt

for name in ("selected", "_tab_f"):
    if name not in globals():
        raise RuntimeError(f"Missing `{name}`; run the earlier cells first.")

idx = np.flatnonzero(selected)
if idx.size == 0:
    raise RuntimeError("selected is empty; run the slice cell first.")

def _col_ci(table, *names):
    m = {n.upper(): n for n in table.dtype.names}
    for name in names:
        if name.upper() in m:
            return m[name.upper()]
    raise KeyError(f"Missing any of {names}; available: {table.dtype.names[:30]}...")

ra_col = _col_ci(_tab_f, "RA")
dec_col = _col_ci(_tab_f, "DEC")
z_col = _col_ci(_tab_f, "Z", "Z_COSMO", "Z_RSD")

ra = np.asarray(_tab_f[ra_col], dtype=np.float64)
dec = np.asarray(_tab_f[dec_col], dtype=np.float64)
z = np.asarray(_tab_f[z_col], dtype=np.float64)

# Segment bounds = tight bounds of the cube-subvolume galaxies, with small padding
SEG_PAD_RA_DEG = 0.25
SEG_PAD_DEC_DEG = 0.25
SEG_PAD_Z = 0.002

ra0, ra1 = float(ra[idx].min() - SEG_PAD_RA_DEG), float(ra[idx].max() + SEG_PAD_RA_DEG)
dec0, dec1 = float(dec[idx].min() - SEG_PAD_DEC_DEG), float(dec[idx].max() + SEG_PAD_DEC_DEG)
z0, z1 = float(z[idx].min() - SEG_PAD_Z), float(z[idx].max() + SEG_PAD_Z)

seg = (ra >= ra0) & (ra <= ra1) & (dec >= dec0) & (dec <= dec1) & (z >= z0) & (z <= z1)

print("RA/DEC/Z segment (derived from cube-subvolume galaxies):")
print(f"  RA  in [{ra0:.4f}, {ra1:.4f}] deg")
print(f"  DEC in [{dec0:.4f}, {dec1:.4f}] deg")
print(f"  {z_col} in [{z0:.5f}, {z1:.5f}]")
print(f"  n_galaxies in segment: {seg.sum():,} / {ra.size:,}")

fig, ax = plt.subplots(figsize=(8.5, 7.0), dpi=200)
sc = ax.scatter(
    ra[seg], dec[seg],
    c=z[seg],
    s=1.0, alpha=0.6,
    cmap="viridis",
    rasterized=True,
)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.set_title("Mock galaxies in survey-space segment (RA/DEC/Z)\nsegment chosen to match cube-subvolume region above")
cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
cb.set_label(z_col)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Requires `_tab_f` (the FITS table after your IN_Y1/IN_Y5 & BOX_INDEX!=-1 filtering)
if "_tab_f" not in globals():
    raise RuntimeError("Missing `_tab_f`; run the FITS-loading cell first.")

def _col_ci(table, *names):
    m = {n.upper(): n for n in table.dtype.names}
    for name in names:
        if name.upper() in m:
            return m[name.upper()]
    raise KeyError(f"Missing any of {names}; available: {table.dtype.names[:30]}...")

ra_col = _col_ci(_tab_f, "RA")
dec_col = _col_ci(_tab_f, "DEC")
z_col = _col_ci(_tab_f, "Z", "Z_COSMO", "Z_RSD")

ra = np.asarray(_tab_f[ra_col], dtype=np.float64)   # deg in [0,360)
dec = np.asarray(_tab_f[dec_col], dtype=np.float64) # deg
zz = np.asarray(_tab_f[z_col], dtype=np.float64)

# ---------------- USER CHOICE: set your segment bounds here ----------------
RA_MIN, RA_MAX = 120.0, 140.0   # deg
DEC_MIN, DEC_MAX = 16.5, 26.7   # deg
Z_MIN, Z_MAX = 0.25, 0.3
# --------------------------------------------------------------------------

# Handle RA wrap-around cleanly (e.g. RA_MIN=350, RA_MAX=20)
if RA_MIN <= RA_MAX:
    m_ra = (ra >= RA_MIN) & (ra <= RA_MAX)
    ra_plot = ra
else:
    m_ra = (ra >= RA_MIN) | (ra <= RA_MAX)
    # unwrap for plotting so the segment is contiguous
    ra_plot = np.where(ra <= RA_MAX, ra + 360.0, ra)

seg = m_ra & (dec >= DEC_MIN) & (dec <= DEC_MAX) & (zz >= Z_MIN) & (zz <= Z_MAX)

print("Selected segment:")
print(f"  RA  in [{RA_MIN}, {RA_MAX}] deg   (wrap={RA_MIN > RA_MAX})")
print(f"  DEC in [{DEC_MIN}, {DEC_MAX}] deg")
print(f"  {z_col} in [{Z_MIN}, {Z_MAX}]")
print(f"  n_galaxies: {int(seg.sum()):,} / {ra.size:,}")

fig, ax = plt.subplots(figsize=(8.5, 7.0), dpi=200)
sc = ax.scatter(
    ra_plot[seg],
    dec[seg],
    c=zz[seg],
    s=1.0,
    alpha=0.6,
    cmap="viridis",
    rasterized=True,
)
ax.set_xlabel("RA [deg]" + (" (unwrapped)" if RA_MIN > RA_MAX else ""))
ax.set_ylabel("DEC [deg]")
ax.set_title("Mock galaxies in chosen RA/DEC/Z segment")
cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
cb.set_label(z_col)
plt.show()

In [ ]:
# --- 3D comoving Cartesian plot for the selected RA/DEC/Z segment ---

import numpy as np
import matplotlib.pyplot as plt

# Needs `seg`, `ra`, `dec`, `zz` from previous cell
for name in ("seg", "ra", "dec", "zz"):
    if name not in globals():
        raise RuntimeError(f"Missing `{name}`; run the RA/DEC/Z segment cell first.")

# Astropy cosmology for comoving distance
from astropy.cosmology import Planck18 as cosmo

ii = np.flatnonzero(seg)
if ii.size == 0:
    raise RuntimeError("Segment is empty (seg.sum()==0).")

# Subsample for speed
RNG = np.random.default_rng(0)
NMAX = 250_000
if ii.size > NMAX:
    ii = RNG.choice(ii, size=NMAX, replace=False)

ra_rad = np.deg2rad(ra[ii])
dec_rad = np.deg2rad(dec[ii])

# Comoving distance in Mpc (not /h). Good enough for visualization.
r = cosmo.comoving_distance(zz[ii]).value

x = r * np.cos(dec_rad) * np.cos(ra_rad)
y = r * np.cos(dec_rad) * np.sin(ra_rad)
z = r * np.sin(dec_rad)

fig = plt.figure(figsize=(9.5, 7.5), dpi=180)
ax = fig.add_subplot(111, projection="3d")
p = ax.scatter(
    x, y, z,
    c=zz[ii],
    s=0.3,
    alpha=0.5,
    cmap="viridis",
    rasterized=True,
)

ax.set_title("Selected RA/DEC/Z segment in 3D comoving Cartesian\ncolor = redshift")
ax.set_xlabel("x [Mpc]")
ax.set_ylabel("y [Mpc]")
ax.set_zlabel("z [Mpc]")

cb = fig.colorbar(p, ax=ax, fraction=0.03, pad=0.02)
cb.set_label("z")

plt.show()

In [ ]:
# --- Interactive Plotly: 3D comoving Cartesian plot for selected RA/DEC/Z segment ---

import numpy as np

for name in ("seg", "ra", "dec", "zz"):
    if name not in globals():
        raise RuntimeError(f"Missing `{name}`; run the RA/DEC/Z segment cell first.")

from astropy.cosmology import Planck18 as cosmo

ii = np.flatnonzero(seg)
if ii.size == 0:
    raise RuntimeError("Segment is empty (seg.sum()==0).")

# Subsample markers only; axis limits use the full mock for consistent context across wedges.
RNG = np.random.default_rng(0)
NMAX = 250_000
if ii.size > NMAX:
    ii = RNG.choice(ii, size=NMAX, replace=False)

ra_rad = np.deg2rad(ra[ii])
dec_rad = np.deg2rad(dec[ii])
r = cosmo.comoving_distance(zz[ii]).value  # Mpc

x = r * np.cos(dec_rad) * np.cos(ra_rad)
y = r * np.cos(dec_rad) * np.sin(ra_rad)
z3 = r * np.sin(dec_rad)

# Scene limits: min/max of the *entire* mock in the same Cartesian frame (not just the segment).
# Optional globals (set in an earlier cell if you like):
#   MOCK_3D_AXIS_RANGES = ((xmin, xmax), (ymin, ymax), (zmin, zmax))  # Mpc; skips the full-array pass
#   MOCK_3D_AXIS_PAD = 0.02  # extra padding as a fraction of each axis span (default 0.02)
#   MOCK_3D_Z_COLOR_RANGE = (z_lo, z_hi)  # colorbar cmin/cmax in redshift; default = full mock zz span
_pad = float(globals().get("MOCK_3D_AXIS_PAD", 0.02))

if "MOCK_3D_AXIS_RANGES" in globals() and globals()["MOCK_3D_AXIS_RANGES"] is not None:
    xr, yr, zr = globals()["MOCK_3D_AXIS_RANGES"]
    xr, yr, zr = tuple(map(float, xr)), tuple(map(float, yr)), tuple(map(float, zr))
else:
    ra_all = np.deg2rad(np.asarray(ra, dtype=np.float64))
    dec_all = np.deg2rad(np.asarray(dec, dtype=np.float64))
    r_all = np.asarray(cosmo.comoving_distance(zz).value, dtype=np.float64)
    cos_d, sin_d = np.cos(dec_all), np.sin(dec_all)
    cos_ra, sin_ra = np.cos(ra_all), np.sin(ra_all)
    x_all = r_all * cos_d * cos_ra
    y_all = r_all * cos_d * sin_ra
    z_all = r_all * sin_d

    def _pad_range(lo: float, hi: float) -> tuple[float, float]:
        span = hi - lo
        if not np.isfinite(span) or span <= 0:
            span = 1.0
        m = _pad * span
        return lo - m, hi + m

    xr = _pad_range(float(np.min(x_all)), float(np.max(x_all)))
    yr = _pad_range(float(np.min(y_all)), float(np.max(y_all)))
    zr = _pad_range(float(np.min(z_all)), float(np.max(z_all)))

if "MOCK_3D_Z_COLOR_RANGE" in globals() and globals()["MOCK_3D_Z_COLOR_RANGE"] is not None:
    z_cmin, z_cmax = map(float, globals()["MOCK_3D_Z_COLOR_RANGE"])
else:
    zz_all = np.asarray(zz, dtype=np.float64)
    z_cmin = float(np.min(zz_all))
    z_cmax = float(np.max(zz_all))
if not np.isfinite(z_cmin) or not np.isfinite(z_cmax) or z_cmax <= z_cmin:
    z_cmax = z_cmin + 1e-6

import plotly.graph_objects as go

fig = go.Figure(
    data=go.Scatter3d(
        x=x, y=y, z=z3,
        mode="markers",
        marker=dict(
            size=2,
            opacity=0.55,
            color=zz[ii],
            cmin=z_cmin,
            cmax=z_cmax,
            colorscale="Viridis",
            colorbar=dict(title="z"),
        ),
    )
)

fig.update_layout(
    title="Selected RA/DEC/Z segment in 3D comoving Cartesian (interactive)<br><sup>color = redshift (mock-wide scale); axes = full mock extent</sup>",
    scene=dict(
        xaxis_title="x [Mpc]",
        yaxis_title="y [Mpc]",
        zaxis_title="z [Mpc]",
        aspectmode="data",
        xaxis=dict(range=list(xr)),
        yaxis=dict(range=list(yr)),
        zaxis=dict(range=list(zr)),
    ),
    template="plotly_white",
    height=750,
)

fig.show()
